# Notebook 06 — Synthesis & KPI Critique

**Goal:** Combine findings from notebooks 03–05 to answer the central question:
*Does the MBTA's official on-time performance metric accurately represent rider experience?*

## Sections
1. Setup & Load
2. Official OTP Calculation
3. Bunching Impact on On-Time Trips (H5 evidence)
4. Rider Experience Score
5. Hypothesis Summary (H1–H5)
6. Headline Stats for Portfolio
7. Save Outputs

## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'

LINE_ORDER  = ['Red Line','Orange Line','Blue Line',
                'Green Line B','Green Line C','Green Line D','Green Line E']
LINE_COLORS = {
    'Red Line':'#DA291C','Orange Line':'#ED8B00','Blue Line':'#003DA5',
    'Green Line B':'#1b7837','Green Line C':'#5aae61',
    'Green Line D':'#a6d96a','Green Line E':'#d9ef8b',
}
print('Imports OK')

In [ ]:
# Load processed outputs from previous notebooks
agg        = pd.read_parquet(PROC_DIR / 'aggregate_full.parquet')
cascade_df = pd.read_parquet(PROC_DIR / 'station_cascade_score.parquet')

# Load ALL monthly strategic parquets (Jan 2024 – May 2026)
# Fix: include headway_trunk_seconds as fallback for Blue & Orange lines
# which have headway_branch_seconds = NaN (single-trunk lines, no branch concept)
OTP_COLS = ['route_id','line','parent_station','stop_timestamp',
             'scheduled_arrival_time','service_date',
             'headway_branch_seconds','headway_trunk_seconds']
frames = []
for fp in sorted((PROC_DIR / 'strategic').glob('*.parquet')):
    frames.append(pd.read_parquet(fp, columns=OTP_COLS))
raw = pd.concat(frames, ignore_index=True)

print(f'aggregate_full       : {len(agg):,} rows')
print(f'station_cascade_score: {len(cascade_df)} stations')
print(f'OTP raw (all months) : {len(raw):,} rows')
print(f'Date range           : {raw["service_date"].min()} → {raw["service_date"].max()}')
print(f'Monthly files loaded : {len(frames)}')

## 2. Official OTP Calculation

In [ ]:
# Convert scheduled_arrival_time (seconds-since-midnight) to Unix timestamp
otp_df = raw.dropna(subset=['stop_timestamp','scheduled_arrival_time']).copy()

otp_df['midnight_unix'] = pd.to_datetime(otp_df['service_date']).apply(
    lambda d: pd.Timestamp(d, tz='America/New_York').timestamp()
)
otp_df['scheduled_unix'] = otp_df['midnight_unix'] + otp_df['scheduled_arrival_time']
otp_df['delay_sec']      = otp_df['stop_timestamp'] - otp_df['scheduled_unix']

# Filter extreme outliers (|delay| > 1 hour = data quality issues)
otp_df = otp_df[otp_df['delay_sec'].abs() <= 3600].copy()

# On-time definition: -60s (not more than 1 min early) to +300s (5 min late)
otp_df['on_time']  = (otp_df['delay_sec'] >= -60) & (otp_df['delay_sec'] <= 300)
otp_df['is_late']  = otp_df['delay_sec'] > 300

# Headway with fallback: branch headway for Green/Red, trunk headway for Blue/Orange
# MBTA LAMP only populates headway_branch_seconds for lines with multiple branches.
# Blue and Orange are single-trunk lines — use headway_trunk_seconds instead.
otp_df['headway']    = otp_df['headway_branch_seconds'].fillna(otp_df['headway_trunk_seconds'])
otp_df['is_bunched'] = otp_df['headway'] < 120

# OTP by line
line_otp = (
    otp_df.groupby('line')
    .agg(
        total_trips = ('on_time','count'),
        on_time_pct = ('on_time','mean'),
        late_pct    = ('is_late','mean'),
        median_delay= ('delay_sec','median'),
    )
    .assign(
        on_time_pct = lambda d: d['on_time_pct']*100,
        late_pct    = lambda d: d['late_pct']*100,
    )
    .round(1)
)

print('=== Official OTP — Jan 2024–May 2026 (threshold: -60s to +300s) ===')
print(line_otp.sort_values('on_time_pct', ascending=False).to_string())

In [ ]:
# Visualise OTP by line
otp_plot = line_otp[line_otp.index.isin(LINE_ORDER)].reindex(
    [l for l in LINE_ORDER if l in line_otp.index]
)

fig, ax = plt.subplots(figsize=(11, 4))
colors = [LINE_COLORS.get(l, '#888') for l in otp_plot.index]
bars = ax.bar(otp_plot.index, otp_plot['on_time_pct'],
               color=colors, edgecolor='white', linewidth=0.8, width=0.6)

for bar, val in zip(bars, otp_plot['on_time_pct']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(80, color='gray', ls='--', lw=1, alpha=0.5, label='80% reference')
ax.set_ylabel('On-Time %')
ax.set_title('Official On-Time Performance by Line — 4 Seasons 2024\n'
              '(on-time = arrived within -60s to +300s of schedule)')
ax.set_xticklabels([l.replace(' Line','\nLine').replace(' Trolley','\nTrolley')
                     for l in otp_plot.index], fontsize=9)
ax.set_ylim(0, 100)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. Bunching Impact on On-Time Trips (H5 Evidence)

**H5:** Official OTP overestimates service quality by masking bunching effects.

Key question: among trips that are *officially on-time*, how many arrived
immediately after a bunching event? These trips look fine in the official stats
but the preceding gap caused severe crowding for passengers on that train.

In [ ]:
# Among officially on-time trips, how many had a preceding bunching event?
ontime_df = otp_df[otp_df['on_time']].copy()
ontime_bunched = ontime_df[ontime_df['is_bunched']]

print('=== H5: Hidden Failures in Official On-Time Trips ===')
print(f'Total on-time trips        : {len(ontime_df):,}')
print(f'On-time but bunched        : {len(ontime_bunched):,}  ({len(ontime_bunched)/len(ontime_df)*100:.1f}%)')
print()

# By line
h5_by_line = (
    ontime_df.groupby('line')
    .apply(lambda x: x['is_bunched'].mean()*100, include_groups=False)
    .round(1)
    .rename('pct_ontime_but_bunched')
    .sort_values(ascending=False)
)
print('=== % of On-Time Trips That Followed a Bunching Event ===')
print(h5_by_line.to_string())
print()

# Flip: among ALL bunching events, what % were classified as on-time?
bunched_df = otp_df[otp_df['is_bunched']]
print('=== Among Bunching Events: On-Time Classification Rate ===')
bunched_otp = (
    bunched_df.groupby('line')
    .agg(total_bunched=('on_time','count'),
         pct_classified_ontime=('on_time','mean'))
    .assign(pct_classified_ontime=lambda d: d['pct_classified_ontime']*100)
    .round(1)
    .sort_values('pct_classified_ontime', ascending=False)
)
print(bunched_otp.to_string())
print()
print('→ A train can be "officially on-time" even when severely bunched.')
print('  The official metric counts the train, not the waiting experience.')

## 4. Rider Experience Score

In [ ]:
# ── Inputs ──────────────────────────────────────────────────────────────────
# 1. Official OTP by line (from all monthly files, computed above)
# 2. Bunching rate by line (from otp_df with headway fallback)
# 3. Cascade impact by line — FIXED: use network_edges for correct station→line mapping
#    Previous bug: lines.str.split(',').str[0] mapped Government Center (out_degree=25)
#    to 'Blue Line' because 'Blue' happened to be listed first in the column.
#    Fix: use network_edges (which has correct route_id per edge) + include ALL stations
#    (not just out_degree > 0) so the mean reflects expected cascade per random trip.

# Bunching rate: use otp_df which already has the headway fallback applied
bunching_by_line = (
    otp_df[otp_df['headway'].notna()]
    .groupby('line')
    .agg(be=('is_bunched','sum'), nh=('headway','count'))
    .assign(bunching_rate=lambda d: d['be'] / d['nh'] * 100)
    [['bunching_rate']]
    .round(2)
)

# Cascade impact: use network_edges for correct line membership
edges = pd.read_parquet(PROC_DIR / 'network_edges.parquet')
route_to_line_e = {
    'Red':'Red Line','Orange':'Orange Line','Blue':'Blue Line',
    'Green-B':'Green Line B','Green-C':'Green Line C',
    'Green-D':'Green Line D','Green-E':'Green Line E',
}
edges['line_e'] = edges['route_id'].map(route_to_line_e)

station_line_pairs = (edges[['from_station','line_e']]
                      .drop_duplicates()
                      .rename(columns={'from_station':'parent_station','line_e':'line'}))

station_cascade = station_line_pairs.merge(
    cascade_df[['parent_station','out_degree']], on='parent_station', how='left'
).fillna({'out_degree': 0})

# Mean over ALL stations per line (including out_degree=0)
# → reflects expected cascade exposure for a random trip on that line
cascade_impact = (
    station_cascade.groupby('line')['out_degree']
    .mean()
    .rename('avg_cascade_outdegree')
    .round(2)
)

# ── Build comparison table ────────────────────────────────────────────────────
score_df = line_otp[['on_time_pct']].rename(columns={'on_time_pct':'official_otp'})
score_df = score_df.join(bunching_by_line).join(cascade_impact)
score_df = score_df[score_df.index.isin(LINE_ORDER)].reindex(
    [l for l in LINE_ORDER if l in score_df.index]
)

# Normalise cascade (0–5 scale) as penalty
max_cascade = score_df['avg_cascade_outdegree'].max()
score_df['cascade_penalty'] = (score_df['avg_cascade_outdegree'] / max_cascade * 5).round(1)

# Rider Experience Score = OTP − bunching_rate − cascade_penalty, clamped [0,100]
score_df['rider_score'] = (
    score_df['official_otp']
    - score_df['bunching_rate']
    - score_df['cascade_penalty'].fillna(0)
).clip(0, 100).round(1)

score_df['gap'] = (score_df['official_otp'] - score_df['rider_score']).round(1)

print('=== Official OTP vs Rider Experience Score (Jan 2024–May 2026) ===')
print(score_df[['official_otp','bunching_rate','cascade_penalty','rider_score','gap']].to_string())
print()
print('✅ All lines computed. cascade_penalty bug fixed:')
print('   Old: lines.str[0] mapped Gov Center (out_degree=25) → Blue Line → penalty=5.0')
print('   Fix: network_edges route_id + mean over ALL stations per line')
print()
print('Formula: Rider Score = Official OTP − bunching_rate − cascade_penalty')

In [ ]:
# Visualise: side-by-side OTP vs Rider Score (all lines)
plot_df = score_df.dropna(subset=['rider_score'])

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(plot_df))
w = 0.35

bars1 = ax.bar(x - w/2, plot_df['official_otp'], w,
                label='Official OTP', color='#4575b4', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + w/2, plot_df['rider_score'],   w,
                label='Rider Experience Score', color='#d73027', alpha=0.85, edgecolor='white')

# Annotate gap
for i, (_, row) in enumerate(plot_df.iterrows()):
    gap = row['official_otp'] - row['rider_score']
    if gap > 0.5:
        ax.annotate(f'−{gap:.0f}', xy=(i, row['rider_score']),
                     xytext=(0, -14), textcoords='offset points',
                     ha='center', fontsize=8, color='#d73027', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 100)
ax.set_title('Official OTP vs Rider Experience Score — Jan 2024–May 2026\n'
              'Gap = bunching penalty + cascade penalty  (all lines now included)')
ax.legend(fontsize=10)
ax.axhline(50, color='gray', ls=':', lw=0.8, alpha=0.5)
plt.tight_layout()
plt.show()
print(f'Lines shown: {list(plot_df.index)}')

## 5. Hypothesis Summary (H1–H5)

In [ ]:
hypothesis_data = [
    {
        'hypothesis': 'H1: Bunching clusters at transfer stations',
        'result': 'Partially Supported',
        'evidence': 'Transfer stations (Kenmore #6, Park St #2) have highest betweenness '
                    'but bunching dominated by outer Green D (Beaconsfield 10.2%, '
                    'Brookline Village 10.2%). Early morning layover artifact inflates outer-D rate.',
        'key_number': 'Outer D bunching 6.19% vs transfer stations ~3.4%',
    },
    {
        'hypothesis': 'H2: Larger headway gap → longer dwell (crowding signal)',
        'result': 'Supported',
        'evidence': 'M3 regression (stop + hour FE): coef=0.083, p≈0. '
                    'At high-ridership stations (above p75): coef=0.128 (+54% lift). '
                    'Effect is real but diluted by low-ridership suburban stops.',
        'key_number': 'coef=0.128 at busy stations: headway 2× → dwell +9.2%',
    },
    {
        'hypothesis': 'H3: Surface stops have shorter dwell (fare evasion proxy)',
        'result': 'Not Supported — direction reversed',
        'evidence': 'Surface median 48s > underground 46s (Mann-Whitney p≈0, N=3.88M). '
                    'Mixed-traffic friction at surface stops outweighs any fare-gate speedup. '
                    'Fare evasion hypothesis not supported by dwell data alone.',
        'key_number': 'Surface 48s vs Underground 46s (+4.3%)',
    },
    {
        'hypothesis': 'H4: High-centrality stations cascade delays across the network',
        'result': 'Supported',
        'evidence': 'Granger causality: Kenmore out_degree=32, Copley=28, Govt Center=25 '
                    '(after filtering terminus stations). Case study 2022-08-10: '
                    'cascade Beaconsfield (6am) → Copley (9am) → Park St (1pm).',
        'key_number': 'Kenmore delays Granger-cause 32 downstream stations',
    },
    {
        'hypothesis': 'H5: Official OTP overestimates service quality',
        'result': 'Supported',
        'evidence': 'Bunched trains classified as on-time: Blue Line 51.9%, Orange 42.2%, '
                    'Green B 24.8%, Green C 22.8%, Green D 20.4%. '
                    'Rider Experience Score lower than official OTP across all lines. '
                    'Green D has the largest gap.',
        'key_number': 'Green D: official 26.3% → rider score 15.1% (gap −11.2 pts); '
                      'Green E −9.6 pts, Green C −8.9 pts, Green B −8.8 pts; '
                      'Blue Line: 51.9% of bunching events still "on-time" (highest in system)',
    },
]

hyp_df = pd.DataFrame(hypothesis_data)
print('=== Hypothesis Summary ===')
for _, row in hyp_df.iterrows():
    print(f'\n{row["hypothesis"]}')
    print(f'  Result  : {row["result"]}')
    print(f'  Evidence: {row["evidence"]}')
    print(f'  Number  : {row["key_number"]}')

## 6. Headline Stats for Portfolio

In [ ]:
print('=' * 60)
print('HEADLINE STATS FOR PORTFOLIO')
print('=' * 60)
print()

# 1. Official OTP range
otp_min = score_df['official_otp'].min()
otp_max = score_df['official_otp'].max()
otp_worst_line = score_df['official_otp'].idxmin()
print(f'1. Official OTP range: {otp_min:.0f}% ({otp_worst_line}) to {otp_max:.0f}%')

# 2. Rider score gap
score_df['gap'] = score_df['official_otp'] - score_df['rider_score']
worst_gap_line = score_df['gap'].idxmax()
worst_gap = score_df['gap'].max()
print(f'2. Largest OTP vs Rider Score gap: {worst_gap:.0f} pts ({worst_gap_line})')

# 3. Bunching
worst_bunching = bunching_by_line['bunching_rate'].idxmax()
worst_rate = bunching_by_line['bunching_rate'].max()
print(f'3. Worst bunching: {worst_bunching} at {worst_rate:.1f}%')

# 4. Super-spreader
top_spreader = cascade_df.sort_values('out_degree', ascending=False).iloc[0]
print(f'4. Top super-spreader: {top_spreader["stop_name"]} '
      f'(out_degree={int(top_spreader["out_degree"])})')

# 5. Green Line structural vulnerability
top_bc = cascade_df.sort_values('betweenness', ascending=False).iloc[0]
print(f'5. Most structurally critical station: {top_bc["stop_name"]} '
      f'(betweenness={top_bc["betweenness"]:.3f})')

# 6. H2 effect at busy stations
print(f'6. Crowding signal: at high-ridership stations, '
      f'headway 2× longer → dwell +9.2% (coef=0.128)')

# 7. Case study
print(f'7. Worst disruption day: 2022-08-10 '
      f'(12.7% system bunching, cascade from outer Green D to Park St over 7 hrs)')

print()
print('KEY STORY:')
print('  The MBTA reports on-time performance by counting individual train arrivals.')
print('  This misses two hidden failure modes:')
print('  (1) Bunched trains arrive on-time but passengers endure severe crowding')
print('      because the preceding gap caused passenger accumulation.')
print('  (2) A single delay at Kenmore or Copley cascades to 25-32 other stations')
print('      — a ripple effect invisible in aggregate OTP statistics.')

## 7. Save Outputs

In [ ]:
# Save final comparison scores
score_df.reset_index().rename(columns={'index':'line'}).to_parquet(
    PROC_DIR / 'final_scores.parquet', index=False
)
print(f'Saved final_scores.parquet')

# Save hypothesis summary
hyp_df[['hypothesis','result','key_number']].to_csv(
    PROC_DIR / 'hypothesis_summary.csv', index=False
)
print(f'Saved hypothesis_summary.csv')
print()
print('=== Final Score Table ===')
print(score_df[['official_otp','bunching_rate','rider_score','gap']].to_string())

## Summary

### Core Finding
Official on-time performance metrics systematically overstate service quality on the MBTA,
particularly on the Green Line, by failing to account for:
1. **Bunching**: trains arrive on-time but preceded by large gaps that cause overcrowding
2. **Cascade**: delays at structurally critical stations (Kenmore, Copley) propagate
   to 25–32 downstream stations, amplifying disruption beyond what OTP records

### Hypothesis Results
| | H1 | H2 | H3 | H4 | H5 |
|--|----|----|----|----|----|
| **Result** | Partial | ✅ | ❌ Reversed | ✅ | ✅ |

### Next Steps
- **ML layer**: Cascade prediction model (Logistic Regression on headway features)
- **Streamlit app**: Interactive cascade simulator
- **Claude API**: Natural language anomaly explainer
- **Web presentation**: `web/` folder with embedded Plotly charts